# 1. Introducción

**Problema industrial:** Pipeline de mantenimiento predictivo con IA para priorización de activos.

**Activo analizado:** Flota de bombas PUMP101 y PUMP102 — mantenimiento integrado.

**Origen de datos:** Series PI multivariable + historial de fallas + scoring de riesgo.

**Objetivo del análisis:** Estimar RUL (vida útil remanente) y generar ranking de intervención (Pareto).


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Pipeline: features → modelo RUL → scoring de riesgo.

In [ ]:
wide = df_good.pivot_table(index="Timestamp", columns="Tag", values="Value", aggfunc="mean").dropna()
wide = wide.reset_index()

# RUL simulado: función decreciente de vibración y temperatura
vib = wide["PUMP101.VIBRATION_RMS"].fillna(wide["PUMP101.VIBRATION_RMS"].mean())
temp = wide["PUMP101.BEARING_TEMP"].fillna(wide["PUMP101.BEARING_TEMP"].mean())
rul = np.clip(500 - 80 * vib - 2 * temp + np.random.default_rng(42).normal(0, 10, len(wide)), 10, 500)

feature_cols = [c for c in wide.columns if c != "Timestamp"]
X = wide[feature_cols].fillna(wide[feature_cols].mean())
y = rul

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rul_model = GradientBoostingRegressor(random_state=42)
rul_model.fit(X_train, y_train)
rul_pred = rul_model.predict(X_test)

ultimo = X.iloc[[-1]]
rul_actual = float(rul_model.predict(ultimo)[0])
riesgo = max(0, min(100, (1 - rul_actual / 500) * 100))

alertas = pd.DataFrame({
    "Activo": ["PUMP101", "PUMP102", "MILL201"],
    "RUL_horas_est": [rul_actual, rul_actual * 1.2, rul_actual * 0.8],
    "Riesgo_%": [riesgo, riesgo * 0.7, riesgo * 1.1],
}).sort_values("Riesgo_%", ascending=False)
resultados_export = alertas
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, rul_pred, alpha=0.5, color="steelblue")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_xlabel("RUL real (h)")
axes[0].set_ylabel("RUL predicho (h)")
axes[0].set_title("Predicción de vida útil remanente")
axes[0].grid(True, alpha=0.3)

alertas_sorted = alertas.sort_values("Riesgo_%", ascending=False)
cum = alertas_sorted["Riesgo_%"].cumsum() / alertas_sorted["Riesgo_%"].sum() * 100
axes[1].bar(alertas_sorted["Activo"], alertas_sorted["Riesgo_%"], color="coral", label="Riesgo %")
ax2 = axes[1].twinx()
ax2.plot(alertas_sorted["Activo"], cum, "ko-", label="Pareto acumulado")
axes[1].set_title("Priorización de alertas — Pareto")
axes[1].set_ylabel("Riesgo %")
ax2.set_ylabel("% acumulado")


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="Alertas", index=False)
    pd.DataFrame({"RUL_actual_h": [rul_actual], "Riesgo_pct": [riesgo]}).to_excel(writer, sheet_name="Scoring", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

PUMP101 concentra el mayor riesgo según RUL estimado. Ejecutar inspección predictiva en las próximas 72 h y actualizar plan maestro de mantenimiento con la priorización Pareto generada.
